# Import Data


In [ ]:
# Import Library yang Diperlukan
!pip install transformers
!pip install pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import string
import warnings
from tqdm.auto import tqdm
import torch
from transformers import BertTokenizer, BertForSequenceClassification, get_scheduler
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Skripsi/makan_siang_gratis_10.csv')
df

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,2.003730e+18,Wed Dec 24 23:59:15 +0000 2025,0,@DS_yantie Lah itu MBG apakabar??? Pak pak,2.003980e+18,NaN,DS_yantie,in,NaN,0,0,0,https://x.com/undefined/status/200397881329920...,7.998670e+17,NaN
1,2.003980e+18,Wed Dec 24 23:58:53 +0000 2025,0,MBG anak sehat Indonesia hebat #FaktaIndonesia...,2.003980e+18,https://pbs.twimg.com/media/G8-QBmJb0AUqtl5.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/200397871919997...,1.615860e+18,NaN
2,2.003980e+18,Wed Dec 24 23:58:23 +0000 2025,0,Pentingnya program MBG untuk masa depan #Fakta...,2.003980e+18,https://pbs.twimg.com/media/G8-P6E2bQAAUbq-.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/200397859238939...,1.615860e+18,NaN
3,2.003980e+18,Wed Dec 24 23:58:09 +0000 2025,0,MBG adalah langkah nyata membangun generasi se...,2.003980e+18,https://pbs.twimg.com/media/G8-P26JbYAA7nnM.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/200397853604726...,1.615860e+18,NaN
4,2.003960e+18,Wed Dec 24 23:58:09 +0000 2025,1,@komaria_cocom MBG anakku loh kak bikin ngelus...,2.003980e+18,NaN,komaria_cocom,in,NaN,0,0,0,https://x.com/undefined/status/200397853383682...,1.971780e+18,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7799,1.928020e+18,Thu May 29 09:29:27 +0000 2025,0,Naahh!! Dengan adanya program MBG anak-anak ga...,1.928020e+18,https://pbs.twimg.com/media/GsG0v5IaUAQG8Iu.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/192802085395617...,9.926921e+08,NaN
7800,1.927670e+18,Thu May 29 09:29:10 +0000 2025,0,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,1.928020e+18,NaN,ranie__14,in,NaN,0,0,0,https://x.com/undefined/status/192802078443534...,9.380510e+17,NaN
7801,1.928020e+18,Thu May 29 09:28:24 +0000 2025,0,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,1.928020e+18,https://pbs.twimg.com/ext_tw_video_thumb/19280...,NaN,in,NaN,0,0,0,https://x.com/undefined/status/192802059203455...,1.262160e+18,NaN
7802,1.927890e+18,Thu May 29 09:28:12 +0000 2025,0,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,1.928020e+18,NaN,txtdrimedia,in,NaN,0,0,0,https://x.com/undefined/status/192802054057857...,1.590840e+18,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7804 entries, 0 to 7803
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   conversation_id_str      7804 non-null   float64
 1   created_at               7804 non-null   object 
 2   favorite_count           7804 non-null   int64  
 3   full_text                7804 non-null   object 
 4   id_str                   7804 non-null   float64
 5   image_url                1326 non-null   object 
 6   in_reply_to_screen_name  4685 non-null   object 
 7   lang                     7804 non-null   object 
 8   location                 0 non-null      float64
 9   quote_count              7804 non-null   int64  
 10  reply_count              7804 non-null   int64  
 11  retweet_count            7804 non-null   int64  
 12  tweet_url                7804 non-null   object 
 13  user_id_str              7804 non-null   float64
 14  username                

# Data Cleaning


In [ ]:
df.drop(['conversation_id_str','created_at','favorite_count','id_str','image_url','in_reply_to_screen_name','lang','location','quote_count','reply_count','retweet_count','tweet_url','user_id_str','username'],axis=1,inplace=True)
df

,full_text
0,@DS_yantie Lah itu MBG apakabar??? Pak pak
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...
2,Pentingnya program MBG untuk masa depan #Fakta...
3,MBG adalah langkah nyata membangun generasi se...
4,@komaria_cocom MBG anakku loh kak bikin ngelus...
...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...


In [ ]:
#Data Duplicate
duplicates = df.duplicated(subset='full_text')
(duplicates.sum())

np.int64(46)

In [ ]:
duplicate_rows = df[df.duplicated(subset='full_text', keep=False)]
(duplicate_rows)

,full_text
836,Sukseskan program MBG
837,Sukseskan program MBG
3720,SAATNYA TOLAK MBG...
3725,SAATNYA TOLAK MBG...
4159,Anak kenyang belajar pun tenang. MBG bukan se...
...,...
6606,STAF Khusus Kepala Badan Gizi Nasional (BGN) b...
6637,Bersama Dukung Program MBG #FaktaIndonesia #Ma...
6951,Bersama Dukung Program MBG #FaktaIndonesia #Ma...
7525,Inilah sebabnya gue prefer pendidikan gratis d...


In [ ]:
df.drop_duplicates(subset='full_text', inplace=True)
(df.count())
df

,full_text
0,@DS_yantie Lah itu MBG apakabar??? Pak pak
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...
2,Pentingnya program MBG untuk masa depan #Fakta...
3,MBG adalah langkah nyata membangun generasi se...
4,@komaria_cocom MBG anakku loh kak bikin ngelus...
...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...


In [ ]:
#Cleaning Data
def cleaning(text):
    text = re.sub(r'@\w+(?:_\w+)*\b', '', text) # menghapus mention
    text = re.sub(r'#\w+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
    text = re.sub(r'r\$\w*', '', text) # menghapus karakter dollar '$'
    text = re.sub(r'[^\x00-\x7F]+', '', text) # menghapus emoji dan karakter non-ASCII
    text = re.sub(r'[^A-Za-z0-9 ]', '', text) # membersihkan teks dengan menghapus semua karakter selain huruf, angka, dan spasi.
    text = re.sub(r'\s+', ' ', text).strip() # mengganti multiple spaces ke single spaces
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

df['text_clean']= df['full_text'].apply(cleaning)

In [ ]:
df

,full_text,text_clean
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...
...,...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...,Naahh Dengan adanya program MBG anakanak gaj p...
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,yang dimaksudkan itu program MBG di daerah yan...
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...


# Case Folding

In [ ]:
#CaseFolding
def case_folding(text):
    text = text.lower()
    return text
df['text_casefolding'] = df['text_clean'].apply(case_folding)
df

,full_text,text_clean,text_casefolding
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...
...,...,...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...,Naahh Dengan adanya program MBG anakanak gaj p...,naahh dengan adanya program mbg anakanak gaj p...
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,yang dimaksudkan itu program MBG di daerah yan...,yang dimaksudkan itu program mbg di daerah yan...
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...,pertemuan prabowomacron bahas isu mbg hingga a...
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...,yg dikasih mbg jutaan siswa kalangan sdsma buk...


In [ ]:
display(df[['text_clean', 'text_casefolding']])

,text_clean,text_casefolding
0,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak
1,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat
2,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan
3,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat
4,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...
...,...,...
7799,Naahh Dengan adanya program MBG anakanak gaj p...,naahh dengan adanya program mbg anakanak gaj p...
7800,yang dimaksudkan itu program MBG di daerah yan...,yang dimaksudkan itu program mbg di daerah yan...
7801,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...,pertemuan prabowomacron bahas isu mbg hingga a...
7802,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...,yg dikasih mbg jutaan siswa kalangan sdsma buk...


# Tokenizing

In [ ]:
!pip install nltk
!pip install Sastrawi

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download data NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('all')
nltk.download('punkt_tab')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 8.3 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_d

True

In [ ]:
df['text_token'] = df['text_casefolding'].apply(lambda x: word_tokenize(x))
df

,full_text,text_clean,text_casefolding,text_token
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak,"[lah, itu, mbg, apakabar, pak, pak]"
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]"
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]"
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener..."
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b..."
...,...,...,...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...,Naahh Dengan adanya program MBG anakanak gaj p...,naahh dengan adanya program mbg anakanak gaj p...,"[naahh, dengan, adanya, program, mbg, anakanak..."
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,yang dimaksudkan itu program MBG di daerah yan...,yang dimaksudkan itu program mbg di daerah yan...,"[yang, dimaksudkan, itu, program, mbg, di, dae..."
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...,pertemuan prabowomacron bahas isu mbg hingga a...,"[pertemuan, prabowomacron, bahas, isu, mbg, hi..."
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...,yg dikasih mbg jutaan siswa kalangan sdsma buk...,"[yg, dikasih, mbg, jutaan, siswa, kalangan, sd..."


In [ ]:
display(df[['text_casefolding', 'text_token']])

,text_casefolding,text_token
0,lah itu mbg apakabar pak pak,"[lah, itu, mbg, apakabar, pak, pak]"
1,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]"
2,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]"
3,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener..."
4,mbg anakku loh kak bikin ngelus dada buat hari...,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b..."
...,...,...
7799,naahh dengan adanya program mbg anakanak gaj p...,"[naahh, dengan, adanya, program, mbg, anakanak..."
7800,yang dimaksudkan itu program mbg di daerah yan...,"[yang, dimaksudkan, itu, program, mbg, di, dae..."
7801,pertemuan prabowomacron bahas isu mbg hingga a...,"[pertemuan, prabowomacron, bahas, isu, mbg, hi..."
7802,yg dikasih mbg jutaan siswa kalangan sdsma buk...,"[yg, dikasih, mbg, jutaan, siswa, kalangan, sd..."


# Normalisasi

In [ ]:
# Baca file CSV yang berisi kolom slang dan formal
# Load kamus slang dari file CSV
slang_dictionary1 = pd.read_excel('/content/drive/MyDrive/Skripsi/kamus_alay.xlsx')
slang_dictionary2 = pd.read_csv('/content/drive/MyDrive/Skripsi/kamus-normalisasi.csv')

# Gabungkan kamus slang
slang_dict1 = pd.Series(slang_dictionary1['kata_baku'].values,index=slang_dictionary1['tidak_baku']).to_dict()
slang_dict2 = pd.Series(slang_dictionary2['kata_baku'].values,index=slang_dictionary2['tidak_baku']).to_dict()

slang_dict = {**slang_dict1, **slang_dict2}

# Fungsi normalisasi
def normalisasi(text):
  if text is not None:
    hasil = [slang_dict[word] if word in slang_dict else word for word in text]
    return hasil
  else:
    return []

df['text_normalize'] = df['text_token'].apply(normalisasi)
df

,full_text,text_clean,text_casefolding,text_token,text_normalize
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak,"[lah, itu, mbg, apakabar, pak, pak]","[lah, itu, mbg, apakabar, pak, pak]"
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]"
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]","[pentingnya , program, mbg, untuk, masa, depan]"
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, adalah, langkah, nyata, membangun, gener..."
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b...","[mbg, anakku, kok, kak, buat, ngelus, dada, bu..."
...,...,...,...,...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...,Naahh Dengan adanya program MBG anakanak gaj p...,naahh dengan adanya program mbg anakanak gaj p...,"[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, dengan, adanya, program, mbg, anakanak..."
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,yang dimaksudkan itu program MBG di daerah yan...,yang dimaksudkan itu program mbg di daerah yan...,"[yang, dimaksudkan, itu, program, mbg, di, dae...","[yang, dimaksudkan, itu, program, mbg, di, dae..."
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...,pertemuan prabowomacron bahas isu mbg hingga a...,"[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, hi..."
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...,yg dikasih mbg jutaan siswa kalangan sdsma buk...,"[yg, dikasih, mbg, jutaan, siswa, kalangan, sd...","[yang, dikasih, mbg, jutaan, siswa, kalangan, ..."


In [ ]:
display(df[['text_token', 'text_normalize']])

,text_token,text_normalize
0,"[lah, itu, mbg, apakabar, pak, pak]","[lah, itu, mbg, apakabar, pak, pak]"
1,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]"
2,"[pentingnya, program, mbg, untuk, masa, depan]","[pentingnya , program, mbg, untuk, masa, depan]"
3,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, adalah, langkah, nyata, membangun, gener..."
4,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b...","[mbg, anakku, kok, kak, buat, ngelus, dada, bu..."
...,...,...
7799,"[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, dengan, adanya, program, mbg, anakanak..."
7800,"[yang, dimaksudkan, itu, program, mbg, di, dae...","[yang, dimaksudkan, itu, program, mbg, di, dae..."
7801,"[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, hi..."
7802,"[yg, dikasih, mbg, jutaan, siswa, kalangan, sd...","[yang, dikasih, mbg, jutaan, siswa, kalangan, ..."


# Stopword Removal

In [ ]:
# get stopword indonesia
list_stopwords = stopwords.words('indonesian')

# append additional stopword
list_stopwords.extend(['klo','kalo', 'amp', 'biar', 'bikin', 'bilang',
                       'gak', 'ga', 'krn', 'nya', 'nih', 'sih',
                       'si', 'tau', 'tdk', 'tuh', 'utk', 'ya',
                       'jd', 'jgn', 'sdh', 'aja', 'n', 't',
                       'nyg', 'hehe', 'pen', 'u', 'nan', 'loh', 'rt',
                       '&amp', 'yah', 'gileee', 'fck', 'up', 'ah'])

# read xls stopword using pandas, specifying the sheet name and using the correct reader
txt_stopword = pd.read_table('/content/drive/MyDrive/Skripsi/stopwords-indonesia.txt', header=None, names=['stopwords'])

# convert stopword string to list & append additional stopword
list_stopwords.extend(txt_stopword['stopwords'][0].split(' '))

# convert list to dictionary (set for faster lookup)
list_stopwords = set(list_stopwords)

#remove stopword pada list token
def stopwords_removal(words):
    return [word for word in words if word not in list_stopwords]

df['text_stopwords'] = df['text_normalize'].apply(stopwords_removal)
df

,full_text,text_clean,text_casefolding,text_token,text_normalize,text_stopwords
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak,"[lah, itu, mbg, apakabar, pak, pak]","[lah, itu, mbg, apakabar, pak, pak]","[mbg, apakabar]"
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]"
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]","[pentingnya , program, mbg, untuk, masa, depan]","[pentingnya , program, mbg]"
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, langkah, nyata, membangun, generasi, sehat]"
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b...","[mbg, anakku, kok, kak, buat, ngelus, dada, bu...","[mbg, anakku, kak, ngelus, dada, salak, susu, ..."
...,...,...,...,...,...,...
7799,Naahh!! Dengan adanya program MBG anak-anak ga...,Naahh Dengan adanya program MBG anakanak gaj p...,naahh dengan adanya program mbg anakanak gaj p...,"[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, program, mbg, anakanak, gaj, membeli, ..."
7800,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,yang dimaksudkan itu program MBG di daerah yan...,yang dimaksudkan itu program mbg di daerah yan...,"[yang, dimaksudkan, itu, program, mbg, di, dae...","[yang, dimaksudkan, itu, program, mbg, di, dae...","[program, mbg, daerah, terpapar, pencemaran, l..."
7801,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...,pertemuan prabowomacron bahas isu mbg hingga a...,"[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, al..."
7802,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...,yg dikasih mbg jutaan siswa kalangan sdsma buk...,"[yg, dikasih, mbg, jutaan, siswa, kalangan, sd...","[yang, dikasih, mbg, jutaan, siswa, kalangan, ...","[dikasih, mbg, jutaan, siswa, kalangan, sdsma,..."


In [ ]:
display(df[['text_normalize', 'text_stopwords']])

,text_normalize,text_stopwords
0,"[lah, itu, mbg, apakabar, pak, pak]","[mbg, apakabar]"
1,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]"
2,"[pentingnya , program, mbg, untuk, masa, depan]","[pentingnya , program, mbg]"
3,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, langkah, nyata, membangun, generasi, sehat]"
4,"[mbg, anakku, kok, kak, buat, ngelus, dada, bu...","[mbg, anakku, kak, ngelus, dada, salak, susu, ..."
...,...,...
7799,"[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, program, mbg, anakanak, gaj, membeli, ..."
7800,"[yang, dimaksudkan, itu, program, mbg, di, dae...","[program, mbg, daerah, terpapar, pencemaran, l..."
7801,"[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, al..."
7802,"[yang, dikasih, mbg, jutaan, siswa, kalangan, ...","[dikasih, mbg, jutaan, siswa, kalangan, sdsma,..."


# Stemming

In [ ]:
!pip install swifter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=12065167d1b55c1ba789a882916ebd12c200145deecf2540e65aae496917ebd5
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter


In [ ]:
# Aktifkan integrasi tqdm dengan pandas
tqdm.pandas()
# Stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import swifter
# create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# stemmed
def stemmed_wrapper(term):
    return stemmer.stem(term)

term_dict = {}

for document in df['text_token']:
    for term in document:
        if term not in term_dict:
            term_dict[term] = ' '

print(len(term_dict))
print("------------------------")

for term in term_dict:
    term_dict[term] = stemmed_wrapper(term)
    print(term,":" ,term_dict[term])

print(term_dict)
print("------------------------")


# apply stemmed term to dataframe
def get_stemmed_term(document):
    return [term_dict[term] for term in document]

df['text_stemmed'] = df['text_token'].apply(get_stemmed_term)

13464
------------------------
mbg : mbg
apakabar : apakabar
anak : anak
sehat : sehat
indonesia : indonesia
hebat : hebat
program : program
langkah : langkah
nyata : nyata
membangun : bangun
generasi : generasi
anakku : anak
kak : kak
ngelus : ngelus
dada : dada
salak : salak
susu : susu
uht : uht
kotak : kotak
biskuit : biskuit
regal : regal
bungkus : bungkus
telor : telor
fokus : fokus
belajar : ajar
pondasi : pondasi
cerah : cerah
dukung : dukung
mengomong : omong
bersyukur : syukur
pencetus : cetus
nanem : nanem
sawit : sawit
menyebabkan : sebab
bencana : bencana
alam : alam
astagfirullah : astagfirullah
adzim : adzim
benaran : benar
rezim : rezim
anti : anti
kritik : kritik
nirempati : nirempati
bidang : bidang
terima : terima
gagal : gagal
noh : noh
parah : parah
wkwkw : wkwkw
konsisten : konsisten
tone : tone
deaf : deaf
gara-gara : gara-gara
warung : warung
makan : makan
sepi : sepi
sekolah : sekolah
beli : beli
kemarin : kemarin
langsung : langsung
langganan : langgan
jatah :

KeyboardInterrupt: 

In [ ]:
# Normalisasi Negasi
#def convert_negasi(text):
#    text = re.sub("tidak ", 'tidak', text, flags=re.MULTILINE)
#    text = re.sub("jangan ", 'jangan', text, flags=re.MULTILINE)
#    text = re.sub("belum ", 'belum', text, flags=re.MULTILINE)
#    text = re.sub("bukan ", 'bukan', text, flags=re.MULTILINE)
#    text = re.sub("tanpa ", 'tanpa', text, flags=re.MULTILINE)
#    text = re.sub("bukanlah ", 'bukanlah', text, flags=re.MULTILINE)
#    text = re.sub("tak ", 'tak', text, flags=re.MULTILINE)
#    text = re.sub("anti ", 'anti', text, flags=re.MULTILINE)
#    return text
#df['text_stemmed']= df['text_stemmed'].apply(lambda x: convert_negasi(x))
#df

# Data Labeling

In [ ]:
!pip install scikit-learn
from sklearn.metrics import accuracy_score
from google.colab import files

In [ ]:
# Function to ensure all elements in a list are strings and handle potential non-list entries
def clean_and_join(word_list):
    if not isinstance(word_list, list):
        return '' # Return empty string if it's not a list (e.g., NaN)
    cleaned_list = []
    for item in word_list:
        if pd.isna(item):
            cleaned_list.append('') # Replace NaN with empty string
        else:
            cleaned_list.append(str(item))
    return ' '.join(cleaned_list).strip()

# Apply the cleaning and joining function to create the 'text' column
df['text'] = df['text_stopwords'].apply(clean_and_join)

# Drop any rows where 'text' might be empty or problematic after cleaning (optional, depending on desired behavior)
df.replace('', np.nan, inplace=True)
df.dropna(subset=['text'], inplace=True)
df = df.reset_index(drop=True)

display(df.head())

,full_text,text_clean,text_casefolding,text_token,text_normalize,text_stopwords,text
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak,"[lah, itu, mbg, apakabar, pak, pak]","[lah, itu, mbg, apakabar, pak, pak]","[mbg, apakabar]",mbg apakabar
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]",mbg anak sehat indonesia hebat
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]","[pentingnya , program, mbg, untuk, masa, depan]","[pentingnya , program, mbg]",pentingnya program mbg
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, langkah, nyata, membangun, generasi, sehat]",mbg langkah nyata membangun generasi sehat
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b...","[mbg, anakku, kok, kak, buat, ngelus, dada, bu...","[mbg, anakku, kak, ngelus, dada, salak, susu, ...",mbg anakku kak ngelus dada salak susu uht kota...


In [ ]:
print(f"Original dataset size: {len(df)}")

Original dataset size: 7755


In [ ]:
duplicates = df.duplicated(subset='text')
(duplicates.sum())

np.int64(393)

In [ ]:
duplicate_rows = df[df.duplicated(subset='text', keep=False)]
(duplicate_rows)

,full_text,text_clean,text_casefolding,text_token,text_normalize,text_stopwords,text
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]",mbg anak sehat indonesia hebat
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]","[pentingnya , program, mbg, untuk, masa, depan]","[pentingnya , program, mbg]",pentingnya program mbg
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, langkah, nyata, membangun, generasi, sehat]",mbg langkah nyata membangun generasi sehat
5,MBG buat anak lebih fokus belajar #MakanBergiz...,MBG buat anak lebih fokus belajar,mbg buat anak lebih fokus belajar,"[mbg, buat, anak, lebih, fokus, belajar]","[mbg, buat, anak, lebih, fokus, belajar]","[mbg, anak, fokus, belajar]",mbg anak fokus belajar
6,MBG Pondasi masa depan cerah #MakanBergiziGrat...,MBG Pondasi masa depan cerah,mbg pondasi masa depan cerah,"[mbg, pondasi, masa, depan, cerah]","[mbg, pondasi, masa, depan, cerah]","[mbg, pondasi, cerah]",mbg pondasi cerah
...,...,...,...,...,...,...,...
7481,Organisasi pengusaha di Prancis Mouvement des ...,Organisasi pengusaha di Prancis Mouvement des ...,organisasi pengusaha di prancis mouvement des ...,"[organisasi, pengusaha, di, prancis, mouvement...","[organisasi, pengusaha, di, prancis, mouvement...","[organisasi, pengusaha, prancis, mouvement, de...",organisasi pengusaha prancis mouvement des ent...
7618,TNI lahir dari rakyat bekerja untuk rakyat dan...,TNI lahir dari rakyat bekerja untuk rakyat dan...,tni lahir dari rakyat bekerja untuk rakyat dan...,"[tni, lahir, dari, rakyat, bekerja, untuk, rak...","[Tentara Nasional Indonesia , Lahir , dari, ...","[Tentara Nasional Indonesia , Lahir , rakyat...",Tentara Nasional Indonesia Lahir rakyat ra...
7619,TNI lahir dari rakyat bekerja untuk rakyat dan...,TNI lahir dari rakyat bekerja untuk rakyat dan...,tni lahir dari rakyat bekerja untuk rakyat dan...,"[tni, lahir, dari, rakyat, bekerja, untuk, rak...","[Tentara Nasional Indonesia , Lahir , dari, ...","[Tentara Nasional Indonesia , Lahir , rakyat...",Tentara Nasional Indonesia Lahir rakyat ra...
7630,@trsYUNA Dapet MBG,Dapet MBG,dapet mbg,"[dapet, mbg]","[dapat, mbg]",[mbg],mbg


In [ ]:
df.drop_duplicates(subset='text', inplace=True)
(df.count())
df

,full_text,text_clean,text_casefolding,text_token,text_normalize,text_stopwords,text
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak,"[lah, itu, mbg, apakabar, pak, pak]","[lah, itu, mbg, apakabar, pak, pak]","[mbg, apakabar]",mbg apakabar
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]","[mbg, anak, sehat, indonesia, hebat]",mbg anak sehat indonesia hebat
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"[pentingnya, program, mbg, untuk, masa, depan]","[pentingnya , program, mbg, untuk, masa, depan]","[pentingnya , program, mbg]",pentingnya program mbg
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, adalah, langkah, nyata, membangun, gener...","[mbg, langkah, nyata, membangun, generasi, sehat]",mbg langkah nyata membangun generasi sehat
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...,"[mbg, anakku, loh, kak, bikin, ngelus, dada, b...","[mbg, anakku, kok, kak, buat, ngelus, dada, bu...","[mbg, anakku, kak, ngelus, dada, salak, susu, ...",mbg anakku kak ngelus dada salak susu uht kota...
...,...,...,...,...,...,...,...
7750,Naahh!! Dengan adanya program MBG anak-anak ga...,Naahh Dengan adanya program MBG anakanak gaj p...,naahh dengan adanya program mbg anakanak gaj p...,"[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, dengan, adanya, program, mbg, anakanak...","[naahh, program, mbg, anakanak, gaj, membeli, ...",naahh program mbg anakanak gaj membeli makanan...
7751,@ranie__14 @endahm7 @Dandhy_Laksono yang dimak...,yang dimaksudkan itu program MBG di daerah yan...,yang dimaksudkan itu program mbg di daerah yan...,"[yang, dimaksudkan, itu, program, mbg, di, dae...","[yang, dimaksudkan, itu, program, mbg, di, dae...","[program, mbg, daerah, terpapar, pencemaran, l...",program mbg daerah terpapar pencemaran logam b...
7752,Pertemuan Prabowo-Macron Bahas Isu MBG hingga ...,Pertemuan PrabowoMacron Bahas Isu MBG hingga A...,pertemuan prabowomacron bahas isu mbg hingga a...,"[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, hi...","[pertemuan, prabowomacron, bahas, isu, mbg, al...",pertemuan prabowomacron bahas isu mbg alutsista
7753,@txtdrimedia Yg dikasih MBG jutaan siswa kalan...,Yg dikasih MBG jutaan siswa kalangan SDSMA Buk...,yg dikasih mbg jutaan siswa kalangan sdsma buk...,"[yg, dikasih, mbg, jutaan, siswa, kalangan, sd...","[yang, dikasih, mbg, jutaan, siswa, kalangan, ...","[dikasih, mbg, jutaan, siswa, kalangan, sdsma,...",dikasih mbg jutaan siswa kalangan sdsma bukti ...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7362 entries, 0 to 7754
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   full_text         7362 non-null   object
 1   text_clean        7362 non-null   object
 2   text_casefolding  7362 non-null   object
 3   text_token        7362 non-null   object
 4   text_normalize    7362 non-null   object
 5   text_stopwords    7362 non-null   object
 6   text              7362 non-null   object
dtypes: object(7)
memory usage: 460.1+ KB


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import ast

# 1. Load Model & Tokenizer
model_name = "mdhugol/indonesia-bert-sentiment-classification"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Sinkronisasi mapping (0:Positif, 1:Netral, 2:Negatif)
model.config.id2label = {0: "Positif", 1: "Netral", 2: "Negatif"}
model.config.label2id = {"Positif": 0, "Netral": 1, "Negatif": 2}

# 2. Inisialisasi pipeline
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

labels = []
scores = []

# 3. Prediksi per baris menggunakan kolom 'text_stemmed'
print(f"Memulai pelabelan dataset...")
for text in df['text'].astype(str).tolist():
    result = classifier(text, truncation=True)

    if result:
        labels.append(result[0]['label'])
        scores.append(result[0]['score'])
    else:
        labels.append('Netral')
        scores.append(0)

# 4. Mapping balik dari teks ke angka (0, 1, 2)
label_map = {"Positif": 0, "Netral": 1, "Negatif": 2}

df["Label_ID"] = [label_map[label] for label in labels]
df["Confidence_Score"] = scores

# 5. Kolom nama sentimen
df['Sentimen'] = labels

print("Pelabelan selesai.")
display(df[['text', 'Sentimen', 'Label_ID', 'Confidence_Score']].head())

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Memulai pelabelan dataset...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Pelabelan selesai.


,text,Sentimen,Label_ID,Confidence_Score
0,mbg apakabar,Negatif,2,0.503783
1,mbg anak sehat indonesia hebat,Positif,0,0.995272
2,pentingnya program mbg,Netral,1,0.665221
3,mbg langkah nyata membangun generasi sehat,Positif,0,0.528011
4,mbg anakku kak ngelus dada salak susu uht kota...,Netral,1,0.990892


In [ ]:
print(df['Sentimen'].value_counts())

Sentimen
Negatif    3974
Netral     1985
Positif    1403
Name: count, dtype: int64


In [ ]:
data_prep = df.to_csv("data_prepro.csv", index=False)

In [ ]:
data_prep = pd.read_csv("data_prepro.csv")
display(data_prep.head())

,full_text,text_clean,text_casefolding,text_token,text_normalize,text_stopwords,text,Label_ID,Confidence_Score,Sentimen
0,@DS_yantie Lah itu MBG apakabar??? Pak pak,Lah itu MBG apakabar Pak pak,lah itu mbg apakabar pak pak,"['lah', 'itu', 'mbg', 'apakabar', 'pak', 'pak']","['lah', 'itu', 'mbg', 'apakabar', 'pak', 'pak']","['mbg', 'apakabar']",mbg apakabar,2,0.503783,Negatif
1,MBG anak sehat Indonesia hebat #FaktaIndonesia...,MBG anak sehat Indonesia hebat,mbg anak sehat indonesia hebat,"['mbg', 'anak', 'sehat', 'indonesia', 'hebat']","['mbg', 'anak', 'sehat', 'indonesia', 'hebat']","['mbg', 'anak', 'sehat', 'indonesia', 'hebat']",mbg anak sehat indonesia hebat,0,0.995272,Positif
2,Pentingnya program MBG untuk masa depan #Fakta...,Pentingnya program MBG untuk masa depan,pentingnya program mbg untuk masa depan,"['pentingnya', 'program', 'mbg', 'untuk', 'mas...","['pentingnya ', 'program', 'mbg', 'untuk', 'm...","['pentingnya ', 'program', 'mbg']",pentingnya program mbg,1,0.665221,Netral
3,MBG adalah langkah nyata membangun generasi se...,MBG adalah langkah nyata membangun generasi sehat,mbg adalah langkah nyata membangun generasi sehat,"['mbg', 'adalah', 'langkah', 'nyata', 'membang...","['mbg', 'adalah', 'langkah', 'nyata', 'membang...","['mbg', 'langkah', 'nyata', 'membangun', 'gene...",mbg langkah nyata membangun generasi sehat,0,0.528011,Positif
4,@komaria_cocom MBG anakku loh kak bikin ngelus...,MBG anakku loh kak bikin ngelus dada Buat hari...,mbg anakku loh kak bikin ngelus dada buat hari...,"['mbg', 'anakku', 'loh', 'kak', 'bikin', 'ngel...","['mbg', 'anakku', 'kok', 'kak', 'buat', 'ngelu...","['mbg', 'anakku', 'kak', 'ngelus', 'dada', 'sa...",mbg anakku kak ngelus dada salak susu uht kota...,1,0.990892,Netral


In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and validation sets (adjust test_size as needed)
train_df, val_df = train_test_split(data_prep, test_size=0.2, random_state=42)

print("Training set shape:", train_df.shape)
print("Validation set shape:", val_df.shape)

Training set shape: (5889, 10)
Validation set shape: (1473, 10)


In [ ]:
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, EarlyStoppingCallback
from datasets import Dataset
import torch
import os

# ======================================================
# 1. GUNAKAN MODEL YANG LEBIH STABIL (SOLUSI UTAMA)
# ======================================================
# Kita ganti ke model benchmark standar agar tidak ada error LayerNorm
model_checkpoint = "indolem/indobert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Pastikan jumlah label sesuai (3 Label)
# Mapping label agar sinkron
id2label = {0: "Negatif", 1: "Netral", 2: "Positif"}
label2id = {"Negatif": 0, "Netral": 1, "Positif": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# ======================================================
# 2. PREPROCESSING (Sama seperti sebelumnya)
# ======================================================
def preprocess_data(examples):
    # Pastikan truncation=True dan max_length cukup
    tokenized_inputs = tokenizer(examples['text_stopwords'], truncation=True, max_length=128)
    tokenized_inputs['labels'] = examples['Label_ID']
    return tokenized_inputs

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(preprocess_data, batched=True)
val_dataset = val_dataset.map(preprocess_data, batched=True)

# Bersihkan kolom tidak perlu
cols_to_keep = ['input_ids', 'attention_mask', 'labels', 'token_type_ids'] # token_type_ids perlu untuk BERT
cols_to_remove = [c for c in train_dataset.column_names if c not in cols_to_keep]
train_dataset = train_dataset.remove_columns(cols_to_remove)
val_dataset = val_dataset.remove_columns(cols_to_remove)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ======================================================
# 3. TRAINING ARGUMENTS (Disesuaikan untuk stabilitas)
# ======================================================
training_args = TrainingArguments(
    output_dir='./results_fix',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Mulai Training Ulang...")
trainer.train()

# ======================================================
# 4. SOLUSI PERBAIKAN STATE DICT (PENTING!)
# ======================================================
print("Memperbaiki penamaan layer (beta/gamma -> bias/weight)...")

# Ambil bobot terbaik hasil training
best_model_weights = model.state_dict()
fixed_weights = {}

for key, value in best_model_weights.items():
    # Mengubah nama lama IndoLEM ke standar Transformers terbaru
    new_key = key.replace('LayerNorm.beta', 'LayerNorm.bias').replace('LayerNorm.gamma', 'LayerNorm.weight')
    fixed_weights[new_key] = value

# Masukkan kembali bobot yang sudah diperbaiki namanya ke dalam model
model.load_state_dict(fixed_weights)

# Simpan secara permanen
final_path = "./indobert_fine_tuned_final"
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

print(f"✅ Training Selesai & Model Berhasil Disimpan di: {final_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/5889 [00:00<?, ? examples/s]

Map:   0%|          | 0/1473 [00:00<?, ? examples/s]

Mulai Training Ulang...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Memperbaiki penamaan layer (beta/gamma -> bias/weight)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training Selesai & Model Berhasil Disimpan di: ./indobert_fine_tuned_final


In [ ]:
import torch
from transformers import pipeline

# ==========================================
# 1. PERBAIKAN STATE DICT (WAJIB SEBELUM PREDIKSI)
# ==========================================
print("Menyelaraskan bobot model...")
best_weights = model.state_dict()
fixed_weights = {}
for key, value in best_weights.items():
    new_key = key.replace('LayerNorm.beta', 'LayerNorm.bias').replace('LayerNorm.gamma', 'LayerNorm.weight')
    fixed_weights[new_key] = value

model.load_state_dict(fixed_weights)

# Sinkronisasi Mapping (Sesuaikan dengan Label_ID awalmu)
model.config.id2label = {0: "Positif", 1: "Netral", 2: "Negatif"}
model.config.label2id = {"Positif": 0, "Netral": 1, "Negatif": 2}

# ==========================================
# 2. INISIALISASI PIPELINE PREDIKSI
# ==========================================
device = 0 if torch.cuda.is_available() else -1
fine_tuned_classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

updated_labels = []
updated_scores = []

print("Memulai prediksi dataset dengan model hasil fine-tune...")
for text in df['text_stopwords'].astype(str).tolist():
    result = fine_tuned_classifier(text, truncation=True)

    if result:
        updated_labels.append(result[0]['label'])
        updated_scores.append(result[0]['score'])
    else:
        updated_labels.append('Netral')
        updated_scores.append(0)

# ==========================================
# 3. PENYIMPANAN HASIL KE DATAFRAME
# ==========================================
# Pastikan mapping angka sesuai dengan standar yang kita pakai di awal (0=Pos, 1=Net, 2=Neg)
label_map = {"Positif": 0, "Netral": 1, "Negatif": 2}

df["Label_fine_tuned"] = [label_map[label] for label in updated_labels]
df["Confidence_fine_tuned"] = updated_scores
df['sentimen_fine_tuned'] = updated_labels

print("✅ Prediksi Selesai!")
display(df[['text_stopwords', 'sentimen_fine_tuned', 'Label_fine_tuned', 'Confidence_fine_tuned']].head())

Menyelaraskan bobot model...
Memulai prediksi dataset dengan model hasil fine-tune...
✅ Prediksi Selesai!


,text_stopwords,sentimen_fine_tuned,Label_fine_tuned,Confidence_fine_tuned
0,"[mbg, apakabar]",Negatif,2,0.935089
1,"[mbg, anak, sehat, indonesia, hebat]",Positif,0,0.947168
2,"[pentingnya , program, mbg]",Positif,0,0.542193
3,"[mbg, langkah, nyata, membangun, generasi, sehat]",Positif,0,0.905261
4,"[mbg, anakku, kak, ngelus, dada, salak, susu, ...",Netral,1,0.526549


In [ ]:
# Menampilkan jumlah masing-masing sentimen setelah fine-tuning
sentimen_fine_tuned_count = df['sentimen_fine_tuned'].value_counts()
print(sentimen_fine_tuned_count)

sentimen_fine_tuned
Negatif    4004
Netral     1866
Positif    1492
Name: count, dtype: int64


In [ ]:
# Save the DataFrame with the fine-tuned sentiment labels to a new CSV file
df.to_csv("data_prepro_labeled.csv", index=False)

In [ ]:
from google.colab import files

files.download("data_prepro_labeled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>